In [ ]:
# run plip

import subprocess
from pathlib import Path

# list of PDB files
pdb_files = ["3p0g.pdb"]
#pdb_files = ["5ifj_abc_cut.pdb", "5ig7_abc_cut.pdb", "5ijk_acx_cut.pdb"]
base_outdir = Path("plip_results")
base_outdir.mkdir(exist_ok=True)

for pdb_path in pdb_files:
    # setup
    pdb_name = Path(pdb_path).stem 
    target_dir = base_outdir / pdb_name
    target_dir.mkdir(exist_ok=True)
    
    # run PLIP
    result = subprocess.run([
        "plip",
        "-f", pdb_path,
        "-o", str(target_dir),
        "--peptides", "A",  # use peptides arg, Chain C
        "-x",  # Generate XML report for data parsing
        "-y",  # human readable text report
        "-t", # Generate PyMOL .pse session file
    ], capture_output=True, text=True)
    # control
    if result.returncode == 0:
        print(f"Results saved to {target_dir}")
    else:
        print(f"Error processing {pdb_name}:")
        print(result.stderr)

Results saved to plip_results/3p0g


In [2]:
# extract interacting residues, residue numbers and residue types from plip xml report

import xml.etree.ElementTree as ET

def get_interacting_residues(xml_path, peptide_chain="C"):

    tree = ET.parse(xml_path)
    root = tree.getroot()

    interacting_res = set()

    interaction_groups = root.findall(".//interactions/*")

    if not interaction_groups:
        print(f"No interactions found in {xml_path}")
        return []

    for group in interaction_groups:
        for contact in group:

            res_chain = contact.findtext("reschain")
            res_num = contact.findtext("resnr")
            res_type = contact.findtext("restype")

            if not all([res_chain, res_num, res_type]):
                continue

            res_chain = res_chain.strip()
            res_num = res_num.strip()
            res_type = res_type.strip()

            # exclude peptide chain
            if res_chain == peptide_chain:
                continue

            interacting_res.add((res_chain, res_num, res_type))

    # sort numerically if possible
    def sort_key(x):
        chain, num, _ = x
        try:
            return (chain, int(num))
        except ValueError:
            return (chain, num)

    return sorted(interacting_res, key=sort_key)

In [5]:
# test
residues = get_interacting_residues("plip_results/5ijk_acx_cut/5ijk_acx_cut_report.xml")
print(residues) # (reschain, resnr, restype)

[('A', '0', 'ASN'), ('A', '55', 'ARG'), ('A', '57', 'LYS'), ('A', '112', 'ILE'), ('A', '113', 'THR'), ('A', '114', 'ILE'), ('B', '31', 'TYR'), ('B', '36', 'LYS'), ('B', '38', 'TYR'), ('B', '56', 'TRP'), ('B', '107', 'TYR'), ('B', '108', 'TYR'), ('B', '113', 'THR')]


In [6]:
from Bio.PDB import PDBParser

d3to1 = {
    'CYS': 'C', 'ASP': 'D', 'SER': 'S', 'GLN': 'Q', 'LYS': 'K',
    'ILE': 'I', 'PRO': 'P', 'THR': 'T', 'PHE': 'F', 'ASN': 'N',
    'GLY': 'G', 'HIS': 'H', 'LEU': 'L', 'ARG': 'R', 'TRP': 'W',
    'ALA': 'A', 'VAL': 'V', 'GLU': 'E', 'TYR': 'Y', 'MET': 'M'
}


def get_pdb_info(pdb_file):

    parser = PDBParser(QUIET=True)
    structure = parser.get_structure("protein", pdb_file)

    model = structure[0]  # only first model

    chains_data = {}
    res_maps = {}
    ordered_chains = []

    for chain in model:

        ordered_chains.append(chain.id)

        seq_list = []
        mapping = {}

        for res in chain:

            hetero_flag, resnum, icode = res.id
            resname = res.get_resname()

            if resname not in d3to1:
                continue

            seq_index = len(seq_list)

            seq_list.append(d3to1[resname])

            # include insertion code if present
            if icode.strip():
                key = f"{resnum}{icode}"
            else:
                key = str(resnum)

            mapping[key] = seq_index

        chains_data[chain.id] = "".join(seq_list)
        res_maps[chain.id] = mapping

    return chains_data, res_maps, ordered_chains

In [8]:
test = get_pdb_info("5ijk_acx_cut.pdb")
print(test) 

({'A': 'EVQLVESGGGLVKPGGSLRLSCAASGFTFSNAWFNWVRQAPGKGLEWVGRIKTNTDGGTTDYAAPVKGRFTISRDDSKNTLYLQMNSLKTEDTAVYYCTTGEPLVNHITILDYWGQGTLVTVSSA', 'B': 'DIVMTQSPDSLAVSLGERATINCKSSQSVLYSSNNKNYLAWYQQKPGQPPKLLIYWASTRESGVPDRFSGSGSGTDFTLTISSLQAEDVAVYYCQQYYRTPPLTFGGGTKVEIKRT', 'C': 'LQPEQPFP'}, {'A': {'1': 0, '2': 1, '3': 2, '4': 3, '5': 4, '6': 5, '7': 6, '8': 7, '9': 8, '11': 9, '12': 10, '13': 11, '14': 12, '15': 13, '16': 14, '17': 15, '18': 16, '19': 17, '20': 18, '21': 19, '22': 20, '23': 21, '24': 22, '25': 23, '26': 24, '27': 25, '28': 26, '29': 27, '30': 28, '35': 29, '36': 30, '37': 31, '38': 32, '39': 33, '40': 34, '41': 35, '42': 36, '43': 37, '44': 38, '45': 39, '46': 40, '47': 41, '48': 42, '49': 43, '50': 44, '51': 45, '52': 46, '53': 47, '54': 48, '55': 49, '56': 50, '57': 51, '58': 52, '59': 53, '60': 54, '61': 55, '62': 56, '63': 57, '64': 58, '65': 59, '66': 60, '67': 61, '68': 62, '69': 63, '70': 64, '71': 65, '72': 66, '74': 67, '75': 68, '76': 69, '77': 70, '78': 71, '79': 72, '80

In [ ]:
### generate mutants (epitope-aware negatives), understand and comment

from pathlib import Path
from Bio.Seq import Seq
from Bio.SeqRecord import SeqRecord
from Bio.SeqIO import write

def build_full_sequence(chain_dict, ordered_chains):
    return ":".join("".join(chain_dict[cid]) for cid in ordered_chains)


def apply_mutations(chains_data, res_maps, ordered_chains, mutations, pdb_name, label_prefix):
    temp_complex = {cid: list(chains_data[cid]) for cid in ordered_chains}
    applied = []

    for mut_chain, res_num, res_type in mutations:
        res_num = str(res_num)

        if mut_chain not in chains_data or res_num not in res_maps[mut_chain]:
            continue

        idx = res_maps[mut_chain][res_num]
        expected_aa = d3to1.get(res_type.upper(), res_type)
        current_aa = temp_complex[mut_chain][idx]

        if current_aa != expected_aa:
            print(f"[{label_prefix}] Mismatch at {mut_chain}{res_num}: "
                  f"Expected {expected_aa}, found {current_aa}")
            continue

        if current_aa == 'A':
            continue

        temp_complex[mut_chain][idx] = 'A'
        applied.append(f"{mut_chain}{expected_aa}{res_num}A")

    if not applied:
        return None, []

    return build_full_sequence(temp_complex, ordered_chains), applied


def generate_mutant_fastas(pdb_file, interacting_res, output_fasta):
    pdb_name = Path(pdb_file).stem
    chains_data, res_maps, ordered_chains = get_pdb_info(pdb_file)

    mutant_records = []

    # add wild type
    wt_seq = build_full_sequence(chains_data, ordered_chains)
    mutant_records.append(SeqRecord(Seq(wt_seq), id=f"{pdb_name}", description=""))

    # handle duplicates early
    interacting_res = sorted(set(interacting_res))

    # single mutants
    for mutation in interacting_res:
        full_seq, applied = apply_mutations(
            chains_data,
            res_maps,
            ordered_chains,
            [mutation],   # one mutation at a time
            pdb_name,
            "SINGLE"
        )

        if full_seq:
            mut_chain, res_num, res_type = mutation
            expected_aa = d3to1.get(res_type.upper(), res_type)
            header = f"{pdb_name}_{mut_chain}_{expected_aa}{res_num}A"
            mutant_records.append(SeqRecord(Seq(full_seq), id=header, description=""))

    # Global interface mutant
    full_seq_all, applied_all = apply_mutations(
        chains_data,
        res_maps,
        ordered_chains,
        interacting_res,
        pdb_name,
        "all_A"
    )

    if full_seq_all:
        header = f"{pdb_name}_all_A"
        mutant_records.append(SeqRecord(Seq(full_seq_all), id=header, description=""))

    with open(output_fasta, "w") as f:
        write(mutant_records, f, "fasta")

    print(f"Generated {len(mutant_records) - 1} mutants for {pdb_name}")

In [10]:
pdb_files = ["5ifj_abc_cut.pdb", "5ig7_abc_cut.pdb", "5ijk_acx_cut.pdb"]

for pdb in pdb_files:
    # get the xml path
    pdb_stem = Path(pdb).stem
    xml_path = f"plip_results/{pdb_stem}/{pdb_stem}_report.xml"
    
    # extract residues
    res_list = get_interacting_residues(xml_path)
    
    # generate fasta
    fasta_out = f"{pdb_stem}_mutants.fasta"
    generate_mutant_fastas(pdb, res_list, fasta_out)

Generated 10 mutants for 5ifj_abc_cut
Generated 9 mutants for 5ig7_abc_cut
Generated 13 mutants for 5ijk_acx_cut


In [11]:
from Bio.PDB import PDBParser

parser = PDBParser(QUIET=True)
structure = parser.get_structure("protein", "5ifj_abc_cut.pdb")

for model in structure:
    for chain in model:
        if chain.id == "A":
            for res in chain:
                hetflag, resseq, icode = res.id
                if resseq == 111:
                    print("Full residue ID:", res.id)
                    print("Residue name:", res.get_resname())

Full residue ID: (' ', 111, ' ')
Residue name: ASP
Full residue ID: (' ', 111, 'A')
Residue name: CYS


In [12]:
from Bio import SeqIO
from pathlib import Path

# List the FASTA files to combine
fasta_to_combine = ["5ifj_abc_cut_mutants.fasta", "5ig7_abc_cut_mutants.fasta", "5ijk_acx_cut_mutants.fasta"]
master_output_name = "combined_mutants.fasta"

def create_master_fasta(file_list, output_name):
    all_records = []
    
    for fasta_file in file_list:
        if Path(fasta_file).exists():
            # Read the records from the individual file
            records = list(SeqIO.parse(fasta_file, "fasta"))
            all_records.extend(records)
            print(f"Added {len(records)} records from {fasta_file}")
        else:
            print(f"Warning: {fasta_file} not found. Skipping.")

    # Write all collected records to the master file
    with open(output_name, "w") as f:
        SeqIO.write(all_records, f, "fasta")
    
    print(f"\nMaster file created: {output_name} (Total records: {len(all_records)})")


create_master_fasta(fasta_to_combine, master_output_name)

Added 11 records from 5ifj_abc_cut_mutants.fasta
Added 10 records from 5ig7_abc_cut_mutants.fasta
Added 14 records from 5ijk_acx_cut_mutants.fasta

Master file created: combined_mutants.fasta (Total records: 35)


In [ ]:
#awk '/^>/ {if (NR > 1) printf("\n"); printf("%s\n", $0); next;} {printf("%s", $0);} END {printf("\n");}' combined_mutants.fasta > mutants.fasta

In [ ]:
# List of your target PDB files
pdb_files = ["5ifj_abc_cut.pdb", "5ig7_abc_cut.pdb", "5ijk_acx_cut.pdb"]

for pdb in pdb_files:
    print(f"{'-'*30}")
    print(pdb)
    print(f"{'-'*30}")
  
    
    # Get info for this specific PDB
    try:
        chains_data, res_maps, ordered_chains = get_pdb_info(pdb)
        
        # Check the mapping for the first 5 residues of the first chain (usually 'A')
        first_chain = ordered_chains[0]
        first_five = list(res_maps[first_chain].items())[:5]
        
        print(f"Mapping for Chain {first_chain}:")
        print("PDB Number -> Sequence Index")
        for pdb_num, seq_idx in first_five:
            print(f"  Residue {pdb_num} is at index {seq_idx}")
            
    except Exception as e:
        print(f"Error processing {pdb}: {e}")

------------------------------
5ifj_abc_cut.pdb
------------------------------
Mapping for Chain A:
PDB Number -> Sequence Index
  Residue 1 is at index 0
  Residue 2 is at index 1
  Residue 3 is at index 2
  Residue 4 is at index 3
  Residue 5 is at index 4
------------------------------
5ig7_abc_cut.pdb
------------------------------
Mapping for Chain A:
PDB Number -> Sequence Index
  Residue 1 is at index 0
  Residue 2 is at index 1
  Residue 3 is at index 2
  Residue 4 is at index 3
  Residue 5 is at index 4
------------------------------
5ijk_acx_cut.pdb
------------------------------
Mapping for Chain A:
PDB Number -> Sequence Index
  Residue 1 is at index 0
  Residue 2 is at index 1
  Residue 3 is at index 2
  Residue 4 is at index 3
  Residue 5 is at index 4


In [14]:
import subprocess
from Bio import AlignIO

# setup dir
output_dir = Path("alignments")
output_dir.mkdir(exist_ok=True)

# List of mutant fastas
final_fastas = [
    "5ifj_abc_cut_mutants.fasta", 
    "5ig7_abc_cut_mutants.fasta", 
    "5ijk_acx_cut_mutants.fasta"
]

for f in final_fastas:
    out = output_dir / f.replace(".fasta", "_aligned.fasta")
    # Using Muscle v5 syntax
    subprocess.run(["muscle", "-align", f, "-output", out], check=True, stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
    print(f"Aligned: {out}")

Aligned: alignments/5ifj_abc_cut_mutants_aligned.fasta
Aligned: alignments/5ig7_abc_cut_mutants_aligned.fasta
Aligned: alignments/5ijk_acx_cut_mutants_aligned.fasta


In [20]:
from bokeh.layouts import gridplot
from bokeh.models.glyphs import Text, Rect
from bokeh.models import ColumnDataSource, Plot, Grid, Range1d
from bokeh.plotting import figure, show
import numpy as np
from Bio import AlignIO

import panel as pn
import panel.widgets as pnw

pn.extension()


def get_colors(seqs):
    """Make colors for amino acids in sequence"""
    # Create a color dictionary for the 20 standard amino acids
    aa_colors = {
        'A': 'black',  # Alanine
        'C': 'orange',  # Cysteine
        'D': 'blue',  # Aspartic acid
        'E': 'green',  # Glutamic acid
        'F': 'purple',  # Phenylalanine
        'G': 'cyan',  # Glycine
        'H': 'pink',  # Histidine
        'I': 'lightgreen',  # Isoleucine
        'K': 'yellow',  # Lysine
        'L': 'lightblue',  # Leucine
        'M': 'darkgreen',  # Methionine
        'N': 'lightpink',  # Asparagine
        'P': 'grey',  # Proline
        'Q': 'lightgrey',  # Glutamine
        'R': 'darkblue',  # Arginine
        'S': 'darkcyan',  # Serine
        'T': 'lime',  # Threonine
        'V': 'magenta',  # Valine
        'W': 'brown',  # Tryptophan
        'Y': 'red',  # Tyrosine
        '-': 'white'  # Gap
    }

    # Flatten the sequence list into a single list of residues
    text = [i for s in list(seqs) for i in s]

    # Map the sequence letters to their corresponding colors
    colors = [aa_colors.get(i, 'white') for i in text]  # Default to white if unknown amino acid
    return colors


def view_alignment(aln, fontsize="9pt", plot_width=800):
    """Bokeh sequence alignment view for protein sequences"""

    # make sequence and id lists from the aln object
    seqs = [rec.seq for rec in (aln)]
    ids = [rec.id for rec in aln]
    text = [i for s in list(seqs) for i in s]
    colors = get_colors(seqs)

    N = len(seqs[0])  # Length of each sequence
    S = len(seqs)  # Number of sequences
    width = .4

    x = np.arange(1, N+1)
    y = np.arange(0, S, 1)

    # Create a 2D grid of coords from the 1D arrays
    xx, yy = np.meshgrid(x, y)

    # Flatten the arrays
    gx = xx.ravel()
    gy = yy.flatten()

    # Rect coordinates with offset for proper display
    recty = gy + .5
    h = 1 / S

    # Create the ColumnDataSource with all the arrays
    source = ColumnDataSource(dict(x=gx, y=gy, recty=recty, text=text, colors=colors, font_size=[fontsize]*len(text), font_name=["courier"]*len(text)))

    # Plot height (based on number of sequences)
    plot_height = len(seqs) * 15 + 50

    x_range = Range1d(0, N + 1, bounds='auto')
    viewlen = N/3

    # View range for close-up view
    view_range = (0, viewlen)
    tools = "xpan, xwheel_zoom, reset, save"

    # Entire sequence view (no text, with zoom)
    p = figure(title=None, width=plot_width, height=200,
               x_range=x_range, y_range=(0, S), tools=tools,
               min_border=0, toolbar_location='below')
    rects = Rect(x="x", y="recty",  width=1, height=1, fill_color="colors",
                 line_color=None, fill_alpha=0.6)
    p.add_glyph(source, rects)
    p.yaxis.visible = False
    p.grid.visible = False

    # Sequence text view with ability to scroll along x-axis
    p1 = figure(title=None, width=plot_width, height=plot_height,
                x_range=view_range, y_range=ids, tools="xpan,reset",
                min_border=0, toolbar_location='below')  # , lod_factor=1)

    # Pass font size and font name explicitly through ColumnDataSource
    glyph = Text(x="x", y="y", text="text", text_align='center', text_color="black",
                 text_font="font_name", text_font_size="font_size")  # Bind to column names in source
    rects = Rect(x="x", y="recty",  width=1, height=1, fill_color="colors",
                 line_color=None, fill_alpha=0.4)
    p1.add_glyph(source, glyph)
    p1.add_glyph(source, rects)

    p1.grid.visible = False
    p1.xaxis.major_label_text_font_style = "bold"
    p1.yaxis.minor_tick_line_width = 0
    p1.yaxis.major_tick_line_width = 0

    p = gridplot([[p], [p1]], toolbar_location='below')

    # Display the plot directly in the notebook
    return p

In [21]:
# Visualize 5ifj to check the fix
aln = AlignIO.read("alignments/5ifj_abc_cut_mutants_aligned.fasta", "fasta")
show(view_alignment(aln))

In [22]:
from pathlib import Path
from Bio import SeqIO
import sys

def audit_mutants(fasta_path, window=5, log_file=None):
    # setup the directory and full path
    log_dir = Path("log")
    log_dir.mkdir(exist_ok=True)
    
    log_path = log_dir / log_file if log_file else None

    records = list(SeqIO.parse(fasta_path, "fasta"))
    if not records:
        return

    wt_seq = str(records[0].seq)  # original seq as 1st
    
    # redirect stdout
    original_stdout = sys.stdout
    f = open(log_path, 'w') if log_path else None
    if f: 
        sys.stdout = f
    
    try:
        valid_count = 0
        for rec in records[1:]:
            mut_seq = str(rec.seq)
            diffs = [i for i in range(len(wt_seq)) if wt_seq[i] != mut_seq[i]]
            
            if len(diffs) == 1:
                idx = diffs[0]
                status = "VALID" if mut_seq[idx] == 'A' else f"ERROR ({mut_seq[idx]})"
                if mut_seq[idx] == 'A': valid_count += 1
                
                start, end = max(0, idx - window), min(len(wt_seq), idx + window + 1)
                context_wt  = wt_seq[start:idx] + "[" + wt_seq[idx] + "]" + wt_seq[idx+1:end]
                context_mut = mut_seq[start:idx] + "[" + mut_seq[idx] + "]" + mut_seq[idx+1:end]
                
                print(f"Mutation: {rec.id} | {status}")
                print(f"  WT Context:  {context_wt}")
                print(f"  Mut Context: {context_mut}\n")
            else:
                print(f"Mutation: {rec.id} | FAILED (Found {len(diffs)} diffs)\n")
                
        print(f"RESULT: Verified {valid_count}/{len(records)-1} mutants.\n")

    finally:
        # clean up and reset terminal output
        if f:
            sys.stdout = original_stdout
            f.close()
            print(f"Log saved to: {log_path}")

files_to_log = [
    "5ifj_abc_cut_mutants.fasta", 
    "5ig7_abc_cut_mutants.fasta", 
    "5ijk_acx_cut_mutants.fasta",
]

for fasta in files_to_log:
    log_name = fasta.replace(".fasta", "_log.txt")
    audit_mutants(fasta, log_file=log_name)

Log saved to: log/5ifj_abc_cut_mutants_log.txt
Log saved to: log/5ig7_abc_cut_mutants_log.txt
Log saved to: log/5ijk_acx_cut_mutants_log.txt
